# Week 4 — Interview Questions on Modules, Packages & Pip

This notebook gathers the Week 4 interview questions around a single theme: how Python code is packaged, imported, and shared. Nothing you build stays in one file, so knowing how modules talk to each other is not trivia — it is how every real project is organised. The sheet opens with the most infamous interview one-liner of all: `if __name__ == '__main__':`. We explain why every module has a `__name__`, what it becomes when a file runs directly (`'__main__'`) versus when it is imported (its own module name), and how that lets the same file behave as both a library and a runnable script.

From there we walk through the import system itself: why we import at all, the different flavours of the statement (`import math`, `from math import factorial`, `import math as m`, clubbing several imports in one line), and the execution order you must understand — importing a module runs its top-level code, so side effects need the `__name__` guard. We also look at `sys.path`, the list of directories Python searches when it resolves an import; the code cell prints it so you can see the working directory, a standard-library location, and the installed-packages folders. The final questions climb one level up in organisation: **packages** are directories of modules marked by an `__init__.py` file, and **third-party packages** arrive from PyPI through `pip`, the package manager that installs numpy, pandas, and the rest of the ecosystem.

By the end of this sheet you will be able to answer the classic module questions precisely, explain the `__name__` idiom in your sleep, and describe with concrete reasoning how one import is resolved from your code all the way out to the package index. Run each code cell as you read — the printed `sys.path`, the `hello nitish` from the user-written `test` module, and `math.floor(4.3)` output make each concept visible in seconds.

In [ ]:
# 1. What do you mean by __name__ == '__main__'

# 1. Modules and a Little dunder Called __name__

## 1.1 What is a module?

A **module** is simply a `.py` file — a reusable container of Python code that can hold variables, functions, and classes. Every module gets a special dunder variable called `__name__`, and its value depends on *how* the file entered the running session. When you **execute a program directly** (run the notebook cell or `python file.py`), Python runs that file under the reserved name `__main__`, so `__name__` inside it is the string `'__main__'`. When the same file is **imported** by another program, Python runs its code under the file's own name and stores that name in `__name__` instead — a module called `mymod.py` sees `'mymod'`.

This one variable is the key to the entire week. The code cell demonstrates it: `def display()` defines a function, then `display()` runs and prints `hello`, and finally `print(__name__)` prints `__main__` — confirming this cell is being executed directly. In a normal script the same line would read `__main__` when you run it yourself and the module name when another file pulls it in. Everything else in this notebook — import syntax, execution order, the famous `if` guard — is built on top of this single fact.

## `__name__ == '__main__'`

```python
print(__name__)   # '__main__' when run directly
                  # 'module_name' when imported
```

- Every module has a `__name__`
- Run directly → `'__main__'`
- Imported → the name of that module
- Use this to decide: work when run directly, don't when imported


In [1]:
def display():
  print('hello')

display()
print(__name__)

hello
__main__


# 2. Importing Modules

## 2.1 Why import, and how

Programs are rarely one file. **Importing** loads another module's code into your program so you can reuse its functions, classes, and variables. Python lets you write your own modules and use third-party ones with exactly the same syntax, so once you learn the pattern once it works everywhere.

The simplest form is `import <module>`, which loads the module and makes its names available through the module's namespace:

```python
import test           # a user-defined module (.py file)
import math           # a built-in module

test.hello('nitish')  # module.function(...)
print(math.floor(4.3))   # 4
```

The `import` statement does two things at once: it *finds* the module (searching the `sys.path` folders) and it *executes* the module's top-level code so its definitions exist. After that, access happens through dotted names — `math.floor` rather than bare `floor` — which keeps namespaces separate and prevents collisions between different modules that happen to define the same name. The demo cells import `test` and `math` together and call both, proving one import line works for user modules (printing `hello nitish`) and builtins (printing `4`) alike.

## Importing modules

```python
import test                       # user-defined .py
import math                       # builtin
test.hello('nitish')              # module.function
```

- A module = a `.py` file — importing loads its code
- User-defined and builtin use the same syntax
- `module.function()` — access through the namespace
- Importing looks up the folder in `sys.path`


In [1]:
# import a module
import test
test.hello('nitish')

hello nitish


In [3]:
# import multiple module -> user defined + builtin
import math
import test

test.hello('nitish')
print(math.floor(4.3))

hello nitish
4


In [4]:
# show builtin modules

# 3. Import Statement Variations

## 3.1 Four ways to import

Python offers several flavours of `import`, and picking the right one is a style-and-readability interview topic. The notebook shows them side by side:

- `import math` — the full module; use names with a prefix (`math.factorial(5)`).
- `import math, random, test` — several modules clubbed in one line.
- `from math import factorial, floor` — imports *specific names* so you can call `factorial(5)` with no prefix.
- `import math as m` — an alias so you can type `m.factorial(5)` (this is why every project writes `import numpy as np`).
- `from math import factorial as f` — alias for a single imported function.

`from module import name` trades namespacing for convenience: the imported names land directly in your scope. The demo `from math import factorial, floor` prints `factorial(5)` as `120` — but note the very next line calls `ceil(4.8)`, which fails with a `NameError` because `ceil` was *not* imported. That is the classic trap to mention in an interview: `from ... import ...` imports exactly the names you list, nothing more. Aliases, meanwhile, are purely cosmetic renaming — `m.factorial(5)` is `math.factorial(5)` under a shorter brand.

In [ ]:
# Normal
import math
import random

In [4]:
# clubbing together
import math,random,test

## Import variations

```python
import math                      # full module
import math, random, test        # multiple
from math import factorial, floor   # specific names
import math as m                 # alias
from math import factorial as f  # alias for function
```

- `from module import name` — direct access (no prefix)
- `as` — a short alias (numpy → np)
- Specific imports = less memory/typing


In [6]:
# importing specific names from module
from math import factorial,floor
from test import hello

print(factorial(5))
ceil(4.8)

120


NameError: ignored

In [7]:
# renaming modules
import math as m
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


m.factorial(5)

120

In [8]:
from math import factorial as f

f(5)


120

# 4. Order of Execution and sys.path

## 4.1 What runs when you import

An import is not just a pointer being created — it is *code being executed*. When you `import test`, Python runs `test.py` from the top to the bottom: imports at the top, class and function definitions, and any top-level statements. Definitions merely *create* functions; the function bodies do not run until called. But top-level statements with side effects (printing, reading files, running main logic) run immediately — which is why modules want a guard, covered next.

**`sys.path`** is the list of folders Python searches, in order, when resolving an import: the script's own directory, standard-library paths, and the directories of installed third-party packages. The code cell prints it, and you can read the folder list straight out of the output — the working folder first, then `/usr/lib/python3.8` and friends for the built-in library, then `dist-packages` for pip-installed libraries. Understanding `sys.path` answers the evergreen interview question "why did my import fail?": if your file is not in any of those folders, Python cannot find it. You can add folders to the list yourself (e.g. `sys.path.append('/some/folder')`), though installing properly is usually the better fix.

In [9]:
import sys
for p in sys.path:
  print(p)

/content
/env/python
/usr/lib/python38.zip
/usr/lib/python3.8
/usr/lib/python3.8/lib-dynload

/usr/local/lib/python3.8/dist-packages
/usr/lib/python3/dist-packages
/usr/local/lib/python3.8/dist-packages/IPython/extensions
/root/.ipython


# 5. The `if __name__ == '__main__':` Guard

## 5.1 One idiom, two behaviours

Because importing executes a module's top-level code, a module with side effects at the top level would run them *for the importers too* — someone importing your helper would trigger your prints, your file writing, your whole program. That is almost never what you want. The guard fixes it:

```python
if __name__ == '__main__':
    main()   # only runs when this file is executed directly
```

The check says: "if this file is the entry point of the program — its name is `__main__` — run the script logic; but if I have been imported as `mymod`, skip it." Same file, two personalities: a **library** (importable, no side effects beyond definitions) and a **script** (running the program when executed).

This is exactly why the earlier `print(__name__)` cell is revealing: run it directly and you see `__main__`; import the file and you would see its module name, and the guarded block would stay silent. Every serious Python script uses this idiom, and it is one of the most-quoted interview facts in the language — partly because it is trivial to explain, and partly because beginners genuinely confuse the two execution modes until they see the dunder.

## Packages & 3rd party

```python
# package = folder of modules (+ __init__.py)
# 3rd party = community packages via pip
```

- A package = a group of modules (a directory)
- `pip install pkg` — 3rd party from PyPI
- numpy/pandas/plotly are all 3rd party packages
- Import chain: `import package.module`


# 6. Packages

## 6.1 From modules to whole directories

A single module grows into many modules as a project grows, and Python groups them into **packages**. A package is a directory that contains modules and possibly sub-packages, and it is recognised as a package by the presence of an `__init__.py` file inside it. That file may be empty — it is often no more than a marker — or it may hold initialisation code imported whenever the package is imported. The directory structure

```
mypkg/
    __init__.py
    helpers.py
    models.py
```

makes `mypkg` a package and `import mypkg.models` reachable. The dotted path mirrors the folders: `import package.module` tells Python to descend into the directory and load the module's code. Sub-packages nest the same way. This is how the scientific Python universe stays organised: `pandas`, `numpy`, and `matplotlib` are giant packages whose `import numpy as np` lines resolve through exactly the `sys.path` mechanism from Section 4. The `__init__.py` marker is the one-line definition interviewers expect you to know: *a directory becomes a package when it contains* `__init__.py`.

## 6.2 Third-party packages and pip

Beyond the standard library, Python's power comes from **third-party packages** — code written by the community and shared freely. The place they converge is **PyPI** (the Python Package Index), a public repository where anyone can publish a package and anyone can download them. The tool that performs the download-and-install is **pip**, a package-manager utility that ships automatically with Python.

```
pip install numpy pandas matplotlib
```

The one-liner above is how every project gains superpowers: pip downloads the requested packages, resolves their dependencies, and copies them into a directory that sits on `sys.path` — which is why `import numpy as np` works right after installation. The notebook foreshadows exactly that: the `import numpy as np` / `import pandas as pd` / `import matplotlib.pyplot as plt` cell is standard third-party usage, and the `import math` cells show the same syntax for built-ins. Two interview facts always score: pip is installed automatically with Python, and the correct way to distribute your own work is to publish it to PyPI so `pip install yourname` can bring it to other machines.

## Summary

This week's sheet unpacked the import system from the single `__name__` dunder all the way to the package index. You learned that a `.py` file is a module, that `__name__` is `'__main__'` when a file runs directly but the module's own name when it is imported, and that this distinction powers the `if __name__ == '__main__':` guard — one file, dual personality. You saw import in every flavour: full module, multi-import, `from ... import name`, and aliasing with `as`. You learned the execution order (importing runs top-level code, so guard side effects) and how `sys.path` directs the search for a module. Finally you climbed to **packages** — directories marked by `__init__.py` — and to **third-party packages** installed from PyPI by pip. Together these answer the week's interview questions and explain how one `import complex_library` call connects your code to thousands of installed modules.